In [ ]:
# M4.5 -- .shift()
# .shift() moves values up or down in a Series by N positions
# the index does not change
# only the values shift

# Positive Shift -- .shift(1) -- moves values down (forward in time)
# Creates a lag
#  Each row will now show what the previous rows value was
# 

# Negative Shift -- .shift(-1) -- moves values up (backward in time)
# creates a lead
# Each row now shows what the next rows value will be
# used for lookahead calculations

In [1]:
# Exercise 1

# Import Libraries
import pandas as pd
import numpy as np

# Demand Data
df = pd.DataFrame({
    "month" : pd.date_range("2024-01-01", periods=6, freq="ME"),
    "supplier": ["Acme"]*3 + ["GlobalCo"]*3,
    "demand" : [1200, 1450, 1100, 2200, 1950, 2400],
    "spend" : [1500, 2200, 1800, 3100, 2800, 3400]
})

# set the index with datetime
df = df.set_index("month")

print(df)


            supplier  demand  spend
month                              
2024-01-31      Acme    1200   1500
2024-02-29      Acme    1450   2200
2024-03-31      Acme    1100   1800
2024-04-30  GlobalCo    2200   3100
2024-05-31  GlobalCo    1950   2800
2024-06-30  GlobalCo    2400   3400


In [ ]:
# Exercise 2 & 3 -- Basic .shift()

# Exercise 2 — Basic .shift()
# Add three columns using .assign():

# demand_lag1 — previous month's demand using .shift(1)
# demand_lead1 — next month's demand using .shift(-1)
# demand_mom_change — month-over-month change: current demand minus demand_lag1

df = (
    df
    .assign(
        demand_lag1 = lambda x: x["demand"].shift(1),
        demand_lead1 = lambda x: x["demand"].shift(-1),
        demand_mom_pct_ch = lambda x: ((x["demand"] - x["demand_lag1"]) / (x["demand_lag1"]) * 100).round(1) 
    )
)

# Then filter rows where demand_mom_change is negative — months where demand dropped from the prior month
print(df[df["demand_mom_pct_ch"] < 0])



            supplier  demand  spend  demand_lag1  demand_lead1  \
month                                                            
2024-03-31      Acme    1100   1800       1450.0        2200.0   
2024-05-31  GlobalCo    1950   2800       2200.0        2400.0   

            demand_mom_pct_ch  
month                          
2024-03-31              -24.1  
2024-05-31              -11.4  


In [ ]:
# exercise 4

# notice the first row when the supplier changes from Acme to GlobalCo
# the .groupby() function returns a NaN
# .shift(1) moves a rows value down
# because we are grouping by supplier the previous value is aligned to Acme and is irrelevant to Global Co
# so the first row or value for each supplier will appear blank
# this ensures each suppliers lag is calculated independently within its own group

df = (
    df
    .assign(
        supplier_lag1 = lambda x: x.groupby(by="supplier")["demand"].shift(1)
    )
)

print(df[["supplier", "demand", "demand_lag1", "supplier_lag1"]])

            supplier  demand  demand_lag1  supplier_lag1
month                                                   
2024-01-31      Acme    1200          NaN            NaN
2024-02-29      Acme    1450       1200.0         1200.0
2024-03-31      Acme    1100       1450.0         1450.0
2024-04-30  GlobalCo    2200       1100.0            NaN
2024-05-31  GlobalCo    1950       2200.0         2200.0
2024-06-30  GlobalCo    2400       1950.0         1950.0
